# Process weather data

In [1]:
from pathlib import Path
from solhycool_evaluation.utils import preprocess_meteonorm_txt_data
import pandas as pd

%load_ext autoreload
%autoreload 2

data_path: Path = Path("../../data")
meteo_data_file_path: Path = data_path / "datasets/tmy_2005_tabernas.txt"
output_path: Path = Path("../results")

df_env = preprocess_meteonorm_txt_data(meteo_data_file_path)
df_env.head()


2025-02-17 16:45:00.758 | INFO     | solhycool_evaluation.utils:preprocess_meteonorm_txt_data:42 - Pre-processed data saved to ../../data/datasets/tmy_2005_tabernas.csv and tmy_2005_tabernas.h5


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td
time,,,,,,,,,,
2005-01-01 00:00:00+00:00,0,0,0,0,10.7,90,0.0,-144.0,6.7,9.1
2005-01-01 01:00:00+00:00,0,0,0,0,11.6,82,0.0,-163.4,0.5,8.7
2005-01-01 02:00:00+00:00,0,0,0,0,11.3,82,0.0,-124.4,0.5,8.4
2005-01-01 03:00:00+00:00,0,0,0,0,11.2,82,0.0,-105.4,0.3,8.3
2005-01-01 04:00:00+00:00,0,0,0,0,11.0,82,0.0,-93.6,0.2,8.1


In [2]:
# Visualize data
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from phd_visualizations.constants import generate_plotly_config

df_env = pd.read_hdf(meteo_data_file_path.with_suffix(".h5"))

var_ids: list[str] = ["Tamb_C", "HR_pct", "precip_mm"]
var_units: list[str] = ["°C", "%", "mm"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    fig.add_trace(
        go.Scattergl(name=var_id, showlegend=True), 
        hf_x=df_env.index, 
        hf_y=df_env[var_id], 
        # max_n_samples=2_000,
        row=i + 1, col=1
    )
    fig.update_yaxes(title_text=f"{var_id.replace("_", " ")} ({var_unit})", row=i + 1)

fig.update_layout(
    height=650,
    width=800,
    title="<b>Weather variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.08, xanchor="left", x=0),
)

fig

# fig.show(
#     config=generate_plotly_config(
#         fig, figure_name=f"solhycool_env_viz_{df_env.index[0].strftime('%Y%m%d')}"
#     )
# )



FigureWidgetResampler({
    'data': [{'name': '<b style="color:sandybrown">[R]</b> Tamb_C <i style="color:#fc9944">~9h</i>',
              'showlegend': True,
              'type': 'scattergl',
              'uid': '9a8519ac-1b4e-401e-bdef-a88b8b5b1692',
              'x': array([datetime.datetime(2005, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 1, 1, 7, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 1, 1, 13, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2005, 12, 31, 10, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 12, 31, 15, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2005, 12, 31, 23, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([10.7, 10.7, 13.4, ...,  7.2, 11.4,  6. ], shape=(1000,)),
    

In [4]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range
save_figure(fig, f"solhycool_weather_viz_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", figure_path=output_path, formats=["png", "svg"])


2025-02-17 16:09:12.443 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results')]/solhycool_weather_viz_20050101_20051231.png
2025-02-17 16:09:13.900 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results')]/solhycool_weather_viz_20050101_20051231.svg
